# Working with off the shelf models

Before we get into loading and playing around with off the shelf models, there are concepts we need to cover. The most important one is "Checkpoints". You will see papers with codes and *checkpoints* made available. 

## So What is a checkpoint?
(ALREADY COVERED?)

A checkpoint is a snapshot of a model’s parameters at a specific point in training. Training is expensive, fragile, and iterative — checkpoints let us stop, resume, reuse, and compare models.

### using state_dict

This will create a dictionary and save the weights and learnable parameters. This doesn't save the architecture, only the parameters. It's called a state_dict for a reason. It's a "state" of the model and you can later on load it into your architecture and the model will behave as it would at the time of saving the state_dict. 


### saving the whole architecture as a serialized model

With torch.save you can save the whole model as a checkpoint. The model will be saved as a serialized object. And so there are downsides to this:
- It depends on your exact Python class existing in the same environment
- If you rename files/classes or change code, loading can break
- It’s less portable across projects
- It can be unsafe to load pickled objects from untrusted sources


So saving the weights and parameters using state_dict, instantiating a model (as a class) and then loading the parameters into that instantiation is safer.  

There are model zoos where you can download pretrained architectures + weights. Usually, what we do with these models is:

- pick a model name
- load the architecture + pretrained weights
- use the correct preprocessing
- optionally attach a head and fine-tune

so when we use an off-the-shelf model we are downloading/loading and using a checkpoint created by the model builders :D


*We will see an example later in the notebook*

## Vision model ecosystems

### torchvision
PyTorch’s official set of common vision models (ResNet, ViT, ConvNeXt, etc.) and datasets/transforms.
It is best for ResNet-style supervised pretrained models; quick demos.

> pip install torchvision

### Hugging Face transformers
A huge hub for pretrained models + standardized loading APIs.
supports many research models (DINOv2, CLIP variants, ViTs), consistent checkpoint + config handling

> pip install transformers

Hugging face usually separates AutoModel (the network) and AutoImageProcessor (preprocessing)

### timm
A very large collection of vision models + pretrained weights used heavily in research. It's popular for its breadth; fast iteration; supports many architectures not in torchvision.

> pip install timm

**These three ecosystems overlap. The same architecture can exist in multiple places, but with different checkpoints and preprocessing. Always check which “family” you’re using.**


## Using off-the-shelf models

Note: this is for models provided in the 3 ecosystems mentioned above. 

- choose your model source (torchvision, hugging face, timm)
- choose a model you want to use and make sure that you know what the name assigned to it is in the source you are using
- load the model + weights
- Get the correct preprocessing (preprocessing can be as simple as a resizing or it can be complicated. It really depends on the model you are using)
- Run a forward pass and inspect output shapes
- Attach a head / freeze / fine-tune (next notebook) (will be in the next notebook)

### Torchvision

You can find all the available models through torchvision by checking torchvision.models!
Here, we will use torchvision to load a ResNet (residual network)

We can load a resnet model and train it from scratch OR we can load the weights from pretrained checkpoint and load our resnet model with those weights. 

With torchvision, we can load the architecture of the model (to train it from scratch) and we can also load checkpoints. In the case of resnet50, we can create an architecture, initialized with random weights using resnet50. We can also load weights from certain checkpoints (trained on different datasets etc). 

REMINDER: For inference, not training, you need to put the model in eval mode. When you put a model in eval mode, you are essentially telling pytorch that I'm using the model for inference not training. And when you do that you are asking pytorch to change layer behaviour. **Importantly** model.eval changes the Dropout and BatchNorm layers behaviour (Check the training notebook)

You also need to apply specific transformations to your images (loaded using PIL). These transformations can be as simple as resizing. The off-the-shelf models also come with a pipeline of transformations that are to be applied to the images


In the next cells you can see how we get a list of all the available architectures and as an example all the available checkpoints for resNet50

***VERY IMPORTANT NOTE***

Torchvision documents all available pretrained checkpoints for each model on their website.
You can find them here: https://docs.pytorch.org/vision/stable/models.html

In [1]:
# get a list of all the model architectures available in torchvision
import torchvision.models as models
for model_name in models.__dict__:  
    # only print the model names that are callable (i.e., functions that return a model instance)
    if callable(models.__dict__[model_name]):
        print(model_name)

alexnet
AlexNet
AlexNet_Weights
ConvNeXt
ConvNeXt_Tiny_Weights
ConvNeXt_Small_Weights
ConvNeXt_Base_Weights
ConvNeXt_Large_Weights
convnext_tiny
convnext_small
convnext_base
convnext_large
DenseNet
DenseNet121_Weights
DenseNet161_Weights
DenseNet169_Weights
DenseNet201_Weights
densenet121
densenet161
densenet169
densenet201
EfficientNet
EfficientNet_B0_Weights
EfficientNet_B1_Weights
EfficientNet_B2_Weights
EfficientNet_B3_Weights
EfficientNet_B4_Weights
EfficientNet_B5_Weights
EfficientNet_B6_Weights
EfficientNet_B7_Weights
EfficientNet_V2_S_Weights
EfficientNet_V2_M_Weights
EfficientNet_V2_L_Weights
efficientnet_b0
efficientnet_b1
efficientnet_b2
efficientnet_b3
efficientnet_b4
efficientnet_b5
efficientnet_b6
efficientnet_b7
efficientnet_v2_s
efficientnet_v2_m
efficientnet_v2_l
googlenet
GoogLeNet
GoogLeNetOutputs
_GoogLeNetOutputs
GoogLeNet_Weights
Inception3
InceptionOutputs
_InceptionOutputs
Inception_V3_Weights
inception_v3
MNASNet
MNASNet0_5_Weights
MNASNet0_75_Weights
MNASNet1_

In [2]:
# get a list of all the checkpoints available for resnet50 in torchvision
# we will use ResNet50_Weights
import torchvision.models as models
for weight_name in models.ResNet50_Weights.__dict__:  
    # print out all the available checkpoints for resnet50
    print(weight_name)

_generate_next_value_
__module__
_new_member_
_use_args_
_member_names_
_member_map_
_value2member_map_
_unhashable_values_
_member_type_
_value_repr_
__doc__
IMAGENET1K_V1
IMAGENET1K_V2
DEFAULT
__new__


In [ ]:
import torch
from torchvision.models import resnet50, ResNet50_Weights

# you can see that we have imported resnet50 and ResNet50_Weights from torchvision.models. 
# ----------------------------------------------------------------
# +++++++++ restnet50 ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# resnet50 is a function that constructs an architecture of the resnet50 model. 
# you can then train it with your own data or load pretrained weights into it.
# model = resnet50()
# builds the ResNet-50 structure
# initializes weights randomly
# no pretraining involved
# +++++++++ ResNet50_Weights ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# this is not a model
# It is a catalog of pretrained checkpoints + metadata
# “Tell me which pretrained ResNet-50 weight files exist, and how they were trained.”

# 1) Choose the weights "package" (this encodes which checkpoint to load)
# here we are using the default weights for resnet50
# this will download th ResNet50_Weights.IMAGENET1K_V2
# Right now, DEFAULT points to IMAGENET1K_V2
# but what you are saying with DEFAULT is "I want to use the default weights for resnet50, whatever they are"
weights = ResNet50_Weights.DEFAULT

# 2) Create the model and load pretrained weights into it
# if you just do model = resnet50(), it will initialize the model with random weights
# but if you do model = resnet50(weights=weights), 
# it will initialize the model with the pretrained weights specified by the weights variable
# now you can start your fine-tuning process with the pretrained weights as a starting point
# or you can just use the pretrained model for inference without any fine-tuning
# but this is essentially what you need to do to load a pretrained model in torchvision:
model = resnet50(weights=weights)

# 3) Put in eval mode for inference demos (turns off dropout, etc.)
# this puts the model in eval mode. 
# This is about layer behavior.
# when you put the model in eval mode, you are telling the model "We are doing inference, not training."
# This changes the behavior of specific layers: 
# Dropout and BatchNorm layers will behave differently in eval mode vs train mode.
model.eval()

# check out the output with a toy input
dummy = torch.randn(2, 3, 224, 224)  # (B, C, H, W)

with torch.no_grad():
    outputs = model(dummy)

print(outputs.shape)
# shape: (B, num_classes) = (2, 1000) for the default resnet50 pretrained on imagenet

torch.Size([2, 1000])


In [4]:
# Torchvision associates the correct preprocessing with the weights object
preprocess = weights.transforms()
# see the transformations that are applied to the input image before it is fed into the model
# it will resize, crop, normalize, etc. the input image according to the preprocessing that was used during the training of the pretrained weights
print(preprocess)

# preprocess is a callable transform pipeline you apply to PIL images
# input = preprocess(pil_image).unsqueeze(0)  # add batch dimension

# SHOW THEM WHAT HAPPENS IF YOU DON'T PREPROCESS
# Preprocessing is part of the model’s “contract.” If you skip it, performance can drop a lot.
# Skipping can also completely break the code if the model expects a certain input size


ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [5]:
# optional: replacing the final layer for fine-tuning
# model.fc is the final fully connected layer of resnet50
# here we are replacing it with a new linear layer that has the same number of input features 
# but a different number of output features (e.g., 10 for CIFAR-10)
import torch.nn as nn

num_features = model.fc.in_features      # how many features come into the final classifier
model.fc = nn.Linear(num_features, 10)   # CIFAR-10 example: 10 classes


### Hugging face

We show how to use Hugging face with a very popular (and good) transformer encoder: DINO. 

Dino (and many other self-supervised models) is an encoder and is distributed as feature extractors. They output token embeddings, not class logits (unlike ResNets).

In transformers, different models are using different classes. For example, Gemini (DOUBLE CHECK THIS) is an instance of transformers.models.vit.modeling_vit.ViTModel whereas Dino is an instance of transformers.models.dinov2.modeling_dinov2.Dinov2Model. So these are two different classes. The reason these have different classes is because they havee different outputs and different configs. Simply these two are so different that it was not possible to use the same class for both and they are defined as different classes. 

To handle this, when we are loading models from hugging face, we use AutoModel (and AutoImageProcessor). To get an idea on what happens, when you use AutoModel with the name of the model you want, there are a bunch of if and elses that let the AutoModel return the right class and config. 

What we do with hugging face is kinda similar to the torchvision models. However, there are subtle differences which you need to take into account when building your model zoo (if you are going to use different packages). With torchvision, as you saw, we import resnet50 which will give us the architecture, and ResNet50_Weights which will give us weights from different checkpoints (using ResNet50_Weights.DEFAULTS you'll get the "best" one). So you import one thing to get the architecture, and another thing to get the weights from a checkpoint.

In hugging face, we use AutoModel:
> AutoModel.from_pretrained(...) 
loads the **architecture definition** and the **pretrained weights**:
- the architecture: The Python class that defines the layers and forward() computation.
- pretrained weights: are downloaded from a checkpoint


Now what happens when you do encoder = AutoModel.**from_pretrained**("facebook/dinov2-base")?

1. hugging face downloads the config (config.jason). The config file contain info about the architecture like: "model_type": "dinov2", "hidden_size": 768, "num_hidden_layers": 12, "num_attention_heads": 12, etc. 
2. instantiate the architecture using the downloaded config: model = Dinov2Model(config), note that it also gets the specific class corresponding to the model in model_name. At this point, all the parameters are randomly initialized. 
3. download the weights from a checkpoint. It uses state_dict.
4. load the weights into the model: model.load_state_dict(state_dict)
5. the ready model is trained. 

Now what if you just need the architecture and you want to train it from scratch? 

you just instantiate the model from the config. 


### Transformers vs CNNs (like resnet)
as you will see, the ouputs are different. Transformers return a vector per patch with dim = embedding_dim. ResNet however, returns a spatial map of feautues (B, num_features, Height, Width). 

Transformers return a CLS token which as you saw is a globally learnt summary, whereas ResNet doesn't return a cls token (a learnable summary). Instead, we can do a global pooling over H and W
> global_avg_pool(features)  # average over H and W



In [ ]:
# loading Dino

import torch
from transformers import AutoImageProcessor, AutoModel

model_name = "facebook/dinov2-base"

# load the pretrained image processor and model
processor = AutoImageProcessor.from_pretrained(model_name)
encoder = AutoModel.from_pretrained(model_name)

# put the model in eval mode
encoder.eval()

# to see what are the transformations applied by the processor
# print(processor)

# to see the model architecture
# to take a look at the model architecture
# print(encoder)

# what does Dino return? 
# look at this toy example:
dummy = torch.randn(2, 3, 224, 224)  # (B, C, H, W)

with torch.no_grad(): # turning off gradients since we are just doing a forward pass for inference
    outputs = encoder(pixel_values=dummy)

tokens = outputs.last_hidden_state
print(tokens.shape)
# shape is (B, num_patches + 1, hidden_dim)


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 1854.76it/s, Materializing param=layernorm.weight]                                 


torch.Size([2, 257, 768])


In [ ]:
# hugging face also has resnet available:
from transformers import AutoModel
import torch


model_name = "microsoft/resnet-50"
encoder = AutoModel.from_pretrained(model_name)
processor = AutoImageProcessor.from_pretrained(model_name)

# to get a sense of the output
dummy = torch.randn(2, 3, 224, 224)  # (B, C, H, W)

with torch.no_grad():
    outputs = encoder(pixel_values=dummy)

features = outputs.last_hidden_state
print(features.shape)

# shape: (B, num_feature_channels, feature_map_height, feature_map_width) = (2, 2048, 7, 7) for resnet-50

# convert to tokens by flattening the spatial dimensions and transposing
B, C, H, W = features.shape
# first flatten the spatial dimensions (H, W) into one dimension (num_patches), then transpose to get (B, num_patches, C)
# for transpose, you need to specify the dimensions you want to swap.
# here it will be the third dimension (C) and the second dimension (num_patches)
tokens = features.flatten(2).transpose(1, 2)
print(tokens.shape)


# NOTE: when you are using torchvision, the output of the model is usually 
# the final output after the global average pooling and fully connected layer, which is (B, num_classes).
# meaning it is class logits that you can use for classification.
# But when you are using hugging face to load resnet, 
# the output is the feature map before the global average pooling and fully connected layer, 
# which is (B, num_feature_channels, feature_map_height, feature_map_width).
# so the output is not class logits, but rather a feature map that you can use for other purposes (e.g., as input to a transformer, or for object detection, etc.)

Loading weights: 100%|██████████| 318/318 [00:00<00:00, 1614.41it/s, Materializing param=encoder.stages.3.layers.2.layer.2.normalization.weight]              
ResNetModel LOAD REPORT from: microsoft/resnet-50
Key                 | Status     |  | 
--------------------+------------+--+-
classifier.1.bias   | UNEXPECTED |  | 
classifier.1.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([2, 2048, 7, 7])
torch.Size([2, 49, 2048])


In [15]:
# here's an important distinction between transformers like Dino and convolutional models like ResNet:
# Dino returns a sequence of tokens (one per patch + a CLS token) with shape (B, num_patches + 1, hidden_dim)
# ResNet returns a feature map with shape (B, num_feature_channels, feature_map_height, feature_map_width) after the convolutional

# now what if you want to have the output of resnet be a sequence of tokens instead of a feature map? Just like the Dino output?
# 


In [ ]:
# load just the architecture without pretrained weights
from transformers import AutoConfig

config = AutoConfig.from_pretrained("facebook/dinov2-base")
# now instantiate the model with this config to get a randomly initialized model
random_model = AutoModel.from_config(config)

## Timm

In [ ]:
import timm

# if you set pretrained=False, it will load the architecture with random weights instead of pretrained weights
model = timm.create_model("vit_base_patch16_224", pretrained=True)
# put the model in eval mode for inference
model.eval()


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False